# Movie Reviews Classification with simple RNN (Tutorial from Geek4Feek)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

In [2]:
df = pd.read_csv("../datasets/imdb.csv", names=["text", "label"])
df.head()

,text,label
0,review,sentiment
1,One of the other reviewers has mentioned that ...,positive
2,A wonderful little production. <br /><br />The...,positive
3,I thought this was a wonderful way to spend ti...,positive
4,Basically there's a family where a little boy ...,negative


In [3]:
df.shape

(50001, 2)

In [4]:
# Convert to lowercase and tokenized them => break the whole sentence into smaller unit, hold byu a Pandas Series
df['text'] = df['text'].str.lower().str.split()
df.head()

,text,label
0,[review],sentiment
1,"[one, of, the, other, reviewers, has, mentione...",positive
2,"[a, wonderful, little, production., <br, /><br...",positive
3,"[i, thought, this, was, a, wonderful, way, to,...",positive
4,"[basically, there's, a, family, where, a, litt...",negative


In [5]:
# Drop row at index 0
df = df.drop(0)

df['label'].value_counts()

label
positive    25000
negative    25000
Name: count, dtype: int64

In [6]:
# Encode label into numeric
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

In [7]:
df.head()

,text,label
1,"[one, of, the, other, reviewers, has, mentione...",1
2,"[a, wonderful, little, production., <br, /><br...",1
3,"[i, thought, this, was, a, wonderful, way, to,...",1
4,"[basically, there's, a, family, where, a, litt...",0
5,"[petter, mattei's, ""love, in, the, time, of, m...",1


In [8]:
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

In [9]:
# Initializing an empty set (sets automatically handle duplicates)
vocab = set()

# 1. Loop through each row in the 'text' column of the DataFrame
for phrase in df['text']:
    # 2. Each 'phrase' is a list of strings (tokens), so we loop through it
    for word in phrase:
        # 3. Add the individual word to our set
        # If the word is already there, the set just ignores the add command
        vocab.add(word)

# Result: vocab now contains every unique word found in the entire dataset

In [10]:
# Initializing an empty dictionary to store word:index pairs
word_to_idx = {}

# We use enumerate() to get both a counter (idx) and the item (word)
# We use start=1 because 0 is usually reserved for "padding" in RNNs
for idx, word in enumerate(vocab, start=1):
    # Create a dictionary entry
    # Key = the string (word), Value = the integer (idx)
    word_to_idx[word] = idx

# Result: word_to_idx looks like {'python': 1, 'machine': 2, 'learning': 3...} => the rule book for the encoding later on

In [11]:
print(len(vocab))
print(len(word_to_idx))

390931
390931


In [12]:
# Padding so that we add "0" to fill up to the max_length => we can create tensor
# Since RNN is built on matrix multiplication => need fixed size (same size for every rows)
"""
Original Sequences (Numbers from your word_to_idx):
Row 1: [1, 2, 3] (Length 3)
Row 2: [4, 5, 6, 7, 8] (Length 5)

After Padding (using max_length):
Row 1: [1, 2, 3, 0, 0] (Now length 5)
Row 2: [4, 5, 6, 7, 8] (Stays length 5)
"""
max_length = df['text'].str.len().max()

print(max_length)

def encode_and_pad(text):
    encoded = [word_to_idx[word] for word in text]
    return encoded + [0] * (max_length - len(encoded))

train_data['text'] = train_data['text'].apply(encode_and_pad)
test_data['text'] = test_data['text'].apply(encode_and_pad)

2470


In [13]:
max_length

np.int64(2470)

In [14]:
train_data

,text,label
39088,"[337433, 366430, 374804, 110950, 260862, 23407...",0
30894,"[374804, 212832, 145021, 301616, 28980, 387462...",0
45279,"[362513, 285168, 4476, 305139, 71779, 270114, ...",1
16399,"[317332, 334717, 389489, 39543, 228270, 362513...",0
13654,"[95812, 270114, 360721, 374804, 208074, 170073...",0
...,...,...
11285,"[105700, 332493, 266132, 28980, 298797, 274467...",1
44733,"[374804, 266417, 317332, 376540, 384794, 18628...",1
38159,"[61033, 317332, 165389, 313864, 228270, 362513...",0
861,"[317332, 181235, 303179, 27625, 362513, 221455...",1


In [ ]:
# Create Dataset and DataLoader
# The DataLoader is like a pickup truck that takes from the SentimentDataset aka the Warehouse
# with batch and shuffled to optimize memory as well as avoid data leakage (model remember the patterns)

class SentimentDataset(Dataset):
    def __init__(self, data):
        self.texts = data['text'].values
        self.labels = data['label'].values
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        return torch.tensor(text, dtype=torch.long), torch.tensor(label, dtype=torch.long)

train_dataset = SentimentDataset(train_data)
test_dataset = SentimentDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [16]:
# Defining the RNN model

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, output_size):
        super(SentimentRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = self.embedding(x)
        h0 = torch.zeros(1, x.size(0), hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

vocab_size = len(vocab) + 1
embed_size = 128
hidden_size = 128
output_size = 2 
model = SentimentRNN(vocab_size, embed_size, hidden_size, output_size)

In [17]:
# Model Training

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for texts, labels in train_loader:
        outputs = model(texts)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss / len(train_loader):.4f}')

KeyboardInterrupt: 